In [0]:

-- =========================================================================
-- 1. INITIAL DATA EXPLORATION & BASELINE VALIDATION
-- =========================================================================

-- Viewing the table schema and initial rows
SELECT *
FROM patient_readmission.netcare_hospital.patient_readmission_csv
LIMIT 5;

-- Checking previous admissions and number of diagnoses
SELECT 
    `Patient ID`,
    `Previous Admissions`,
    `Number of Diagnoses`
FROM patient_readmission.netcare_hospital.patient_readmission_csv
WHERE `Previous Admissions` > 2
   OR `Number of Diagnoses` > 3
LIMIT 5;

-- Checking unique admission types
SELECT DISTINCT `Admission Type` AS Unique_Admission_Type
FROM patient_readmission.netcare_hospital.patient_readmission_csv;

-- Checking a specific baseline age
SELECT `Age`
FROM patient_readmission.netcare_hospital.patient_readmission_csv
WHERE `Age` = 18
LIMIT 5;


-- =========================================================================
-- 2. CORRECTED METRIC & AGGREGATION ANALYSIS
-- =========================================================================

-- Checking length of stay by unique patients (Prevents row inflation)
SELECT 
    `Length of Stay`,
    COUNT(DISTINCT `Patient ID`) AS Unique_Patients
FROM 
    patient_readmission.netcare_hospital.patient_readmission_csv
WHERE 
    `Length of Stay` > 5
GROUP BY 
    `Length of Stay`
ORDER BY 
    Unique_Patients DESC
LIMIT 10;

-- Checking average blood sugar levels specifically for readmitted patients
SELECT 
    `Readmission`,
    ROUND(AVG(`Blood Sugar Levels`), 2) AS Average_Blood_Sugar
FROM patient_readmission.netcare_hospital.patient_readmission_csv
WHERE `Readmission` = 'Yes'
GROUP BY `Readmission`;

-- Checking number of unique patients with high blood pressure (>140)
SELECT 
    CASE 
        WHEN `Blood Pressure` > 140 THEN 'High BP (>140)'
        ELSE 'Normal/Low BP'
    END AS Blood_Pressure_Status,
    COUNT(DISTINCT `Patient ID`) AS Unique_Patients
FROM patient_readmission.netcare_hospital.patient_readmission_csv
GROUP BY 1;


-- =========================================================================
-- 3. FINAL ENHANCED DASHBOARD VIEW (With Corrected Buckets)
-- =========================================================================
-- This query standardizes your labels to match the dashboard visuals 
-- ('Older Adults' instead of 'Old Folks') and prepares the exact buckets needed.

SELECT 
    `Patient ID`,
    `Gender`,
    `Age`,
    `Length of Stay`,
    `Blood Pressure`,
    `Blood Sugar Levels`,
    `Readmission`,

    -- Age Group (Aligned with dashboard visuals: Young Adult, Older Adults, Senior)
    CASE
        WHEN `Age` < 18 THEN 'Minor'
        WHEN `Age` BETWEEN 18 AND 35 THEN 'Young Adult'
        WHEN `Age` BETWEEN 36 AND 64 THEN 'Older Adults' 
        ELSE 'Senior'
    END AS Age_Group,

    -- Length of Stay Grouping
    CASE
        WHEN `Length of Stay` < 5 THEN 'Short'
        WHEN `Length of Stay` BETWEEN 5 AND 10 THEN 'Medium'
        WHEN `Length of Stay` > 10 THEN 'Long'
        ELSE 'Unknown' 
    END AS Length_Of_Stay_Group,

    -- Diagnoses Complexity Grouping
    CASE
        WHEN `Number of Diagnoses` < 3 THEN 'Low'
        WHEN `Number of Diagnoses` BETWEEN 3 AND 5 THEN 'Medium'
        WHEN `Number of Diagnoses` > 5 THEN 'High'
        ELSE 'Unknown'
    END AS Number_Of_Diagnoses_Group,

    -- Health Risk Profiling: Blood Pressure Group
    CASE
        WHEN `Blood Pressure` < 120 THEN 'Low'
        WHEN `Blood Pressure` BETWEEN 120 AND 140 THEN 'Medium'
        -- Matches dashboard logic where high risk starts mapping near your filters
        WHEN `Blood Pressure` > 140 THEN 'High' 
        ELSE 'Unknown'
    END AS Blood_Pressure_Group,

    -- Blood Sugar Grouping
    CASE
        WHEN `Blood Sugar Levels` < 100 THEN 'Low'
        WHEN `Blood Sugar Levels` BETWEEN 100 AND 125 THEN 'Medium'
        WHEN `Blood Sugar Levels` > 125 THEN 'High'
        ELSE 'Unknown'
    END AS Blood_Sugar_Group

FROM patient_readmission.netcare_hospital.patient_readmission_csv;